In [20]:
#Fixing time series data
import pandas as pd
df = pd.read_csv('messy_sales_data.csv')

df['sale_date_clean'] = pd.to_datetime(df['sale_date'], 
                                       format='mixed', #this make pandas check every row seperately to figure out the format
                                       errors='coerce') #if pandas cannot figure out the format, this will turn it into a missing value

In [ ]:
#OLS Regression
import pandas as pd
import statsmodels.api as sm

#Load the data
df = pd.read_csv('BeerTax.csv') 

#Run a regression model
y = df['fatalpop']
X = df['beertax']
X = sm.add_constant(X)
model1 = sm.OLS(y,X) #define the model
resultsOLS = model1.fit() #fit the model to the data
print(resultsOLS.summary()) #print the results

In [ ]:
#Fixed effects model
from linearmodels.panel.model import PanelOLS #import the function

df = pd.read_csv('BeerTax.csv') #load the data
df = df.set_index(['state', 'year']) #this tells python what the group and time variables are

y = df['fatalpop']
X = df['beertax']
modelFE = PanelOLS(y,X,entity_effects=True)
resultsFE = modelFE.fit()
print(resultsFE.summary)



In [ ]:
#Random effects model
from linearmodels.panel.model import RandomEffects #import the function

df = pd.read_csv('BeerTax.csv') #load the data
df = df.set_index(['state','year']) #this tells python what the group and time variables are

#Create the model. 
y = df['fatalpop']
X = df['beertax']
X = sm.add_constant(X) #note that you still need to add an intercept

modelRE = RandomEffects(y,X)
resultsRE = modelRE.fit() #Fit the model
print(resultsRE.summary) #print the results

In [ ]:
import numpy as np
from scipy import stats

b_fe = resultsFE.params
b_re = resultsRE.params.drop('const', errors='ignore')
cov_fe = resultsFE.cov
cov_re = resultsRE.cov.drop('const', axis=0, errors='ignore').drop('const', axis=1, errors='ignore')

diff = b_fe - b_re
cov_diff = cov_fe - cov_re

stat = diff.T @ np.linalg.inv(cov_diff) @ diff
pval = 1 - stats.chi2.cdf(stat, df=len(diff))

print(f"Hausman stat: {stat:.4f}, p-value: {pval:.4f}")